# ProPainter Remote Worker

Run Cell 1 once per session to mount Drive and install deps.  
Run Cell 2 once to define helpers.  
Run Cell 4 once to verify setup (Drive + ProPainter + GPU).  
Run Cell 3 to start the worker loop — leave it running.

The worker polls `jobs/pending/` every 30 seconds, processes each job with ProPainter, and writes results to `jobs/done/<job_id>/result.mp4`.

In [ ]:
# ── Cell 1 — Setup (run once per session) ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
!pip install -q einops kornia av

DRIVE_ROOT     = '/content/drive/MyDrive/video_agent_jobs'
PENDING_DIR    = f'{DRIVE_ROOT}/jobs/pending'
PROCESSING_DIR = f'{DRIVE_ROOT}/jobs/processing'
DONE_DIR       = f'{DRIVE_ROOT}/jobs/done'
FAILED_DIR     = f'{DRIVE_ROOT}/jobs/failed'
HEARTBEAT      = f'{DRIVE_ROOT}/heartbeat.json'

import os
for d in [PENDING_DIR, PROCESSING_DIR, DONE_DIR, FAILED_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists('/content/ProPainter'):
    !git clone https://github.com/sczhou/ProPainter /content/ProPainter
    !cd /content/ProPainter && pip install -q -r requirements.txt

print('Setup complete')

In [ ]:
# ── Cell 2 — Helpers ──────────────────────────────────────────────────────────
import json, shutil, subprocess, time, re
from datetime import datetime, timezone
from pathlib import Path

def utcnow_str():
    return datetime.now(timezone.utc).isoformat()

def write_heartbeat():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                           capture_output=True, text=True)
        gpu = r.stdout.strip()
    except Exception:
        gpu = 'unknown'
    Path(HEARTBEAT).write_text(json.dumps({
        'status': 'online', 'updated_at': utcnow_str(), 'colab_gpu': gpu
    }))

def write_progress(job_id, frames_done, frames_total):
    Path(PROCESSING_DIR, job_id, 'progress.json').write_text(json.dumps({
        'frames_done': frames_done, 'frames_total': frames_total,
        'updated_at': utcnow_str()
    }))

def check_cancel(job_id):
    return Path(PROCESSING_DIR, job_id, 'cancel.flag').exists()

def run_propainter(job_id, job_dir):
    meta    = json.loads((job_dir / 'job.json').read_text())
    out_dir = f'/content/propainter_out/{job_id}'
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        'python', '/content/ProPainter/inference_propainter.py',
        '--video',  str(job_dir / 'segment.mp4'),
        '--mask',   str(job_dir / 'mask.png'),
        '--output', out_dir,
        '--width',  str(meta.get('width',  400)),
        '--height', str(meta.get('height', 711)),
        '--fp16', '--save_video',
    ]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    frames_total = int(meta.get('fps', 30) * meta.get('duration', 3))

    for line in proc.stdout:
        print(line, end='')
        m = re.search(r'(\d+)/(\d+)', line)
        if m and ('Processing' in line or 'Inpainting' in line):
            write_progress(job_id, int(m.group(1)), int(m.group(2)))
        if check_cancel(job_id):
            proc.terminate()
            return None, 'Cancelled by user'

    proc.wait()
    if proc.returncode != 0:
        return None, f'ProPainter exited {proc.returncode}'

    mp4s = list(Path(out_dir).glob('*.mp4'))
    if not mp4s:
        # ProPainter writes to a subfolder named after the input file stem
        mp4s = list(Path(out_dir).rglob('inpaint_out.mp4'))
    if not mp4s:
        return None, 'No output MP4 found'
    return str(mp4s[0]), None

In [ ]:
# ── Cell 3 — Worker loop (run and leave running) ───────────────────────────────
import time

print('Worker started — watching for jobs...')
write_heartbeat()
last_heartbeat = time.time()

while True:
    if time.time() - last_heartbeat > 60:
        write_heartbeat()
        last_heartbeat = time.time()

    jobs = sorted(Path(PENDING_DIR).iterdir()) if Path(PENDING_DIR).exists() else []
    jobs = [j for j in jobs if j.is_dir()]

    if not jobs:
        time.sleep(30)
        continue

    job_dir  = jobs[0]
    job_id   = job_dir.name
    proc_dir = Path(PROCESSING_DIR) / job_id
    print(f'\n[{utcnow_str()}] Picked up: {job_id}')
    shutil.move(str(job_dir), str(proc_dir))

    try:
        output_mp4, error = run_propainter(job_id, proc_dir)
        if error:
            raise RuntimeError(error)

        done_dir    = Path(DONE_DIR) / job_id
        done_dir.mkdir(parents=True, exist_ok=True)
        result_path = done_dir / 'result.mp4'
        shutil.copy(output_mp4, result_path)

        r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                            '-of', 'csv=p=0', str(result_path)],
                           capture_output=True, text=True)
        duration = float(r.stdout.strip()) if r.stdout.strip() else 0.0

        (done_dir / 'status.json').write_text(json.dumps({
            'status': 'done',
            'output_path': f'jobs/done/{job_id}/result.mp4',
            'duration': duration,
            'completed_at': utcnow_str(),
        }))
        print(f'[{utcnow_str()}] Done — {duration:.1f}s')

    except Exception as e:
        print(f'[{utcnow_str()}] FAILED: {e}')
        failed_dir = Path(FAILED_DIR) / job_id
        failed_dir.mkdir(parents=True, exist_ok=True)
        (failed_dir / 'status.json').write_text(json.dumps({
            'status': 'failed', 'error': str(e), 'failed_at': utcnow_str()
        }))

    finally:
        if proc_dir.exists():
            shutil.rmtree(str(proc_dir), ignore_errors=True)

In [ ]:
# ── Cell 4 — Verify setup (run once after Cell 1) ─────────────────────────────
for folder in [PENDING_DIR, PROCESSING_DIR, DONE_DIR, FAILED_DIR]:
    Path(folder).mkdir(parents=True, exist_ok=True)
    print(f'OK: {folder}')

write_heartbeat()
print(f'Heartbeat written to {HEARTBEAT}')

r = subprocess.run(['python', '/content/ProPainter/inference_propainter.py', '--help'],
                   capture_output=True, text=True)
print('ProPainter:', 'OK' if r.returncode == 0 else f'FAILED:\n{r.stderr[:300]}')

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip())